In [59]:
import sqlite3
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [55]:
import sqlite3
import pandas as pd

DB_PATH = r"../data/inventory.db"

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM vendor_invoice", conn)
conn.close()

df.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None


In [57]:
vendor_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5543 entries, 0 to 5542
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   VendorNumber  5543 non-null   int64         
 1   VendorName    5543 non-null   object        
 2   InvoiceDate   5543 non-null   datetime64[ns]
 3   PONumber      5543 non-null   int64         
 4   PODate        5543 non-null   object        
 5   PayDate       5543 non-null   object        
 6   Quantity      5543 non-null   int64         
 7   Dollars       5543 non-null   float64       
 8   Freight       5543 non-null   float64       
 9   Approval      374 non-null    object        
dtypes: datetime64[ns](1), float64(2), int64(3), object(4)
memory usage: 433.2+ KB


In [61]:
required = ["InvoiceDate", "Dollars", "Freight"]
missing_columns = [col for col in required if col not in vendor_df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

vendor_df = vendor_df.copy()
vendor_df["InvoiceDate"] = pd.to_datetime(vendor_df["InvoiceDate"], errors="coerce")

for col in ["Dollars", "Freight"]:
    vendor_df[col] = pd.to_numeric(vendor_df[col], errors="coerce")

invalid_rows = (
    vendor_df["InvoiceDate"].isna()
    | ~np.isfinite(vendor_df["Dollars"])
    | ~np.isfinite(vendor_df["Freight"])
)

print("Invalid records removed:", invalid_rows.sum())
vendor_df = vendor_df.loc[~invalid_rows].reset_index(drop=True)


Invalid records removed: 0


In [63]:
vendor_df["InvoiceDate"] = pd.to_datetime(vendor_df["InvoiceDate"], errors="coerce")
vendor_df = vendor_df.dropna(subset=["InvoiceDate"])
vendor_df = vendor_df.reset_index(drop=True)

vendor_df.head()

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None


In [65]:
X = df[["Dollars"]]
y = df["Freight"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (5543, 1)
Target shape: (5543,)


In [67]:
cutoff = pd.Timestamp("2024-02-01")


train = vendor_df["InvoiceDate"] < cutoff
test = vendor_df["InvoiceDate"] >= cutoff

X_train = X.loc[train]
X_test = X.loc[test]
y_train = y.loc[train]
y_test = y.loc[test]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

if X_train.empty or X_test.empty:
    raise ValueError("Training or testing data is empty. Check the cutoff date.")

Training rows: 357
Testing rows: 5186
